In [10]:
import time
import os
import re
import json
import subprocess
import numpy as np
from collections import Counter

from experiments import prob_dict, largest_data_transformations
from search_methods.probe_ATE_search import ProbManager
from data_loader import TwinsDataLoader, LalondeDataLoader, ACSDataLoader, IHDPDataLoader, WalmartDataLoader
from experiments import whole_df_ops, LEGAL_OPS_BY_TYPE, LEGAL_FILL_BY_TYPE



In [11]:
df_twins = TwinsDataLoader().load_data()
df_lalonde = LalondeDataLoader().load_data()
df_acs = ACSDataLoader().load_data()
df_IHDP = IHDPDataLoader().load_data()
df_walmart = WalmartDataLoader().load_data()

loaded cached data
loaded cached data
loaded cached data
loaded cached data


In [12]:
common_causes_twins = df_twins.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_lalonde = df_lalonde.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_acs = df_acs.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_ihdp = df_IHDP.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_walmart = df_walmart.columns.difference(["treatment", "outcome"], sort=False).tolist()

In [13]:
ATE_PATTERN = re.compile(r"ATE IS:\s*([-+]?\d*\.?\d+)")
current_notebook_dir = os.getcwd()
main_path = os.path.abspath(os.path.join(current_notebook_dir, "..", "main.py"))

def extract_sequence(output):
    start = output.find("[")
    if start == -1:
        return None

    depth = 0
    end = None
    for i in range(start, len(output)):
        if output[i] == "[":
            depth += 1
        elif output[i] == "]":
            depth -= 1
            if depth == 0:
                end = i + 1
                break

    if end is None:
        return None

    try:
        return json.loads(output[start:end])
    except Exception:
        return None



def get_prob(df, sequence, trans_dict, op_probs, common_causes):
    pm = ProbManager(list(trans_dict.keys()),
                                   common_causes, op_probs, None, whole_df_ops, LEGAL_OPS_BY_TYPE,
                                   legal_fill_by_type=LEGAL_FILL_BY_TYPE, col_types=df.attrs.get('col_types', None),
                                   fill_columns=[col for col in df.columns if df[col].isna().any()])
    return pm.get_sequence_probability(sequence)


def run_experiment(exp_num, df, common_causes, largest_data_transformations, n_runs=10, output_txt=None):
    functions = [
        "llm_zero_shot",
        "llm_few_shot",
        "llm_few_shot_cot"
    ]

    if output_txt is None:
        output_txt = f"exp_{exp_num}_results.txt"

    output_lines = []

    def log_print(msg=""):
        print(msg)
        output_lines.append(str(msg))

    log_print("=" * 60)
    log_print(f" 🚀 STARTING EXPERIMENT: EXP{exp_num}")
    log_print("=" * 60)

    for func in functions:
        exp_arg = f"EXP{exp_num}"

        log_print(f"\n⚙️ Running mode: {func} ({n_runs} repetitions)")
        log_print("-" * 40)

        run_data = []

        for run_idx in range(n_runs):
            start_time = time.time()

            result = subprocess.run(
                ["python", main_path, exp_arg, func],
                capture_output=True,
                text=True
            )

            output = result.stdout
            elapsed = time.time() - start_time
            ate_match = ATE_PATTERN.search(output)

            if ate_match:
                ate = float(ate_match.group(1))
                ate = round(ate, 5)
                sequence = extract_sequence(output)

                run_data.append({
                    "ate": ate,
                    "sequence": sequence,
                    "runtime": elapsed
                })

                log_print(f"  Run {run_idx+1}/{n_runs}: ATE={ate} | Runtime: {elapsed:.2f}s")
            else:
                log_print(f"  Run {run_idx+1}/{n_runs}: ATE not found | Runtime: {elapsed:.2f}s")

            time.sleep(7)

        if run_data:
            ate_values = [x["ate"] for x in run_data]
            runtimes = [x["runtime"] for x in run_data]

            mean_ate = np.mean(ate_values)
            variance_ate = np.var(ate_values, ddof=1) if len(ate_values) > 1 else 0.0
            std_ate = np.std(ate_values, ddof=1) if len(ate_values) > 1 else 0.0
            avg_runtime = np.mean(runtimes)

            ate_counter = Counter(ate_values)
            most_common_ate, ate_freq = ate_counter.most_common(1)[0]
            matching_runs = [x for x in run_data if x["ate"] == most_common_ate]
            chosen_sequence = matching_runs[0]["sequence"] if matching_runs else None

            log_print("\nResults:")
            log_print(f"  ATEs          : {ate_values}")
            log_print(f"  Mean ATE      : {mean_ate:.6f}")
            log_print(f"  Variance      : {variance_ate:.6f}")
            log_print(f"  Std Dev       : {std_ate:.6f}")
            log_print(f"  Avg Runtime   : {avg_runtime:.2f}s")

            log_print("\nMost Frequent ATE:")
            log_print(f"  ATE           : {most_common_ate:.5f}")
            log_print(f"  Frequency     : {ate_freq}/{n_runs}")

            if chosen_sequence:
                log_print("\nCorresponding Sequence:")
                log_print(json.dumps(chosen_sequence, indent=2))

                prob_sequence = tuple(
                    (step["operation"], step["column"])
                    for step in chosen_sequence
                )

                try:
                    sequence_probability = get_prob(
                        df,
                        prob_sequence,
                        largest_data_transformations,
                        prob_dict,
                        common_causes
                    )
                    log_print(f"\nSequence Probability: {sequence_probability}")
                except Exception as e:
                    log_print("\nCould not calculate sequence probability:")
                    log_print(str(e))
            else:
                log_print("\nNo valid sequence extracted")
        else:
            log_print("\nNo valid runs recorded for this mode.")

    log_print("\n" + "=" * 60)
    log_print(f" ✅ FINISHED EXPERIMENT: EXP{exp_num}")
    log_print("=" * 60)

    # Save output to text file
    with open(output_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(output_lines))

    print(f"\n📄 Results saved to: {output_txt}")

In [14]:
run_experiment(
    exp_num=27,
    df=df_lalonde,
    common_causes=common_causes_lalonde,
    largest_data_transformations=largest_data_transformations,
    output_txt='exp27_llm_results.txt'
)

 🚀 STARTING EXPERIMENT: EXP27

⚙️ Running mode: llm_zero_shot (10 repetitions)
----------------------------------------
  Run 1/10: ATE=620.29664 | Runtime: 6.16s
  Run 2/10: ATE=888.70551 | Runtime: 6.20s
  Run 3/10: ATE=1226.22168 | Runtime: 6.11s
  Run 4/10: ATE=620.29664 | Runtime: 6.08s
  Run 5/10: ATE=888.70551 | Runtime: 6.01s
  Run 6/10: ATE=1226.22168 | Runtime: 6.10s
  Run 7/10: ATE=888.70551 | Runtime: 6.12s
  Run 8/10: ATE=620.29664 | Runtime: 6.04s
  Run 9/10: ATE=888.70551 | Runtime: 6.22s
  Run 10/10: ATE=1392.96358 | Runtime: 6.16s

Results:
  ATEs          : [620.29664, 888.70551, 1226.22168, 620.29664, 888.70551, 1226.22168, 888.70551, 620.29664, 888.70551, 1392.96358]
  Mean ATE      : 926.111890
  Variance      : 76027.569224
  Std Dev       : 275.730973
  Avg Runtime   : 6.12s

Most Frequent ATE:
  ATE           : 888.70551
  Frequency     : 4/10

Corresponding Sequence:
[
  {
    "column": "TABLE",
    "operation": "drop_duplicates"
  },
  {
    "column": "age",
 

In [15]:
run_experiment(
    exp_num='27_lal_zero',
    df=df_lalonde,
    common_causes=common_causes_lalonde,
    largest_data_transformations=largest_data_transformations,
    output_txt='exp27_lal_zero_llm_results.txt'
)

 🚀 STARTING EXPERIMENT: EXP27_lal_zero

⚙️ Running mode: llm_zero_shot (10 repetitions)
----------------------------------------
  Run 1/10: ATE=620.29664 | Runtime: 6.48s
  Run 2/10: ATE=620.29664 | Runtime: 6.22s
  Run 3/10: ATE=620.29664 | Runtime: 6.20s
  Run 4/10: ATE=620.29664 | Runtime: 6.26s
  Run 5/10: ATE=620.29664 | Runtime: 6.30s
  Run 6/10: ATE=620.29664 | Runtime: 6.29s
  Run 7/10: ATE=620.29664 | Runtime: 6.27s
  Run 8/10: ATE=620.29664 | Runtime: 6.13s
  Run 9/10: ATE=620.29664 | Runtime: 6.65s
  Run 10/10: ATE=620.29664 | Runtime: 6.33s

Results:
  ATEs          : [620.29664, 620.29664, 620.29664, 620.29664, 620.29664, 620.29664, 620.29664, 620.29664, 620.29664, 620.29664]
  Mean ATE      : 620.296640
  Variance      : 0.000000
  Std Dev       : 0.000000
  Avg Runtime   : 6.31s

Most Frequent ATE:
  ATE           : 620.29664
  Frequency     : 10/10

Corresponding Sequence:
[
  {
    "column": "TABLE",
    "operation": "drop_duplicates"
  },
  {
    "column": "age",
   

In [16]:
run_experiment(
    exp_num=28,
    df=df_acs,
    common_causes=common_causes_acs,
    largest_data_transformations=largest_data_transformations,
    output_txt='exp28_llm_results.txt'
)

 🚀 STARTING EXPERIMENT: EXP28

⚙️ Running mode: llm_zero_shot (10 repetitions)
----------------------------------------
  Run 1/10: ATE=14522.9925 | Runtime: 7.08s
  Run 2/10: ATE=14522.9925 | Runtime: 6.89s
  Run 3/10: ATE=14522.9925 | Runtime: 6.75s
  Run 4/10: ATE=14522.9925 | Runtime: 6.83s
  Run 5/10: ATE=14522.9925 | Runtime: 6.94s
  Run 6/10: ATE=14522.9925 | Runtime: 6.88s
  Run 7/10: ATE=14522.9925 | Runtime: 6.86s
  Run 8/10: ATE=14522.9925 | Runtime: 6.82s
  Run 9/10: ATE=14522.9925 | Runtime: 6.85s
  Run 10/10: ATE=14522.9925 | Runtime: 6.67s

Results:
  ATEs          : [14522.9925, 14522.9925, 14522.9925, 14522.9925, 14522.9925, 14522.9925, 14522.9925, 14522.9925, 14522.9925, 14522.9925]
  Mean ATE      : 14522.992500
  Variance      : 0.000000
  Std Dev       : 0.000000
  Avg Runtime   : 6.86s

Most Frequent ATE:
  ATE           : 14522.99250
  Frequency     : 10/10

Corresponding Sequence:
[
  {
    "column": "TABLE",
    "operation": "drop_duplicates"
  },
  {
    "colu

In [17]:
run_experiment(
    exp_num=35,
    df=df_walmart,
    common_causes=common_causes_walmart,
    largest_data_transformations=largest_data_transformations,
    output_txt='exp35_llm_results.txt'
)

 🚀 STARTING EXPERIMENT: EXP35

⚙️ Running mode: llm_zero_shot (10 repetitions)
----------------------------------------
  Run 1/10: ATE=47384.58618 | Runtime: 7.78s
  Run 2/10: ATE=48896.71071 | Runtime: 7.63s
  Run 3/10: ATE=46059.83263 | Runtime: 7.99s
  Run 4/10: ATE=45361.06821 | Runtime: 8.09s
  Run 5/10: ATE=47579.90451 | Runtime: 8.29s
  Run 6/10: ATE=47579.90451 | Runtime: 8.70s
  Run 7/10: ATE=49405.60512 | Runtime: 9.32s
  Run 8/10: ATE=42691.23161 | Runtime: 8.28s
  Run 9/10: ATE=47384.58618 | Runtime: 7.99s
  Run 10/10: ATE=45361.06821 | Runtime: 8.29s

Results:
  ATEs          : [47384.58618, 48896.71071, 46059.83263, 45361.06821, 47579.90451, 47579.90451, 49405.60512, 42691.23161, 47384.58618, 45361.06821]
  Mean ATE      : 46770.449787
  Variance      : 3849722.288430
  Std Dev       : 1962.070918
  Avg Runtime   : 8.24s

Most Frequent ATE:
  ATE           : 47384.58618
  Frequency     : 2/10

Corresponding Sequence:
[
  {
    "column": "TABLE",
    "operation": "drop_du